In [3]:
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import soundfile as sf


def trim_trailing_silence(
    audio: np.ndarray,
    sample_rate: int,
    top_db: float = 40.0,
    frame_length: int = 2048,
    hop_length: int = 512,
    keep_silence_ms: float = 0.0,
) -> Tuple[np.ndarray, float]:
    """Trim only the silence at the end of a waveform using a frame RMS threshold."""
    if frame_length <= 0 or hop_length <= 0:
        raise ValueError("frame_length and hop_length must be positive integers.")
    if audio.size == 0:
        return audio, 0.0

    envelope = np.max(np.abs(audio), axis=1) if audio.ndim == 2 else np.abs(audio)
    peak_amplitude = float(np.max(envelope))
    if peak_amplitude == 0.0:
        return audio, 0.0

    frame_stats = []
    for frame_start in range(0, envelope.shape[0], hop_length):
        frame_end = min(frame_start + frame_length, envelope.shape[0])
        frame = envelope[frame_start:frame_end]
        if frame.size == 0:
            continue
        frame_rms = float(np.sqrt(np.mean(frame ** 2)))
        frame_stats.append((frame_start, frame_end, frame_rms))

    if not frame_stats:
        return audio, 0.0

    peak_rms = max(frame_rms for _, _, frame_rms in frame_stats)
    if peak_rms == 0.0:
        return audio, 0.0

    rms_threshold = peak_rms * (10.0 ** (-top_db / 20.0))
    sample_threshold = peak_amplitude * (10.0 ** (-top_db / 20.0))

    last_non_silent_sample = None
    for _, frame_end, frame_rms in frame_stats:
        if frame_rms > rms_threshold:
            last_non_silent_sample = frame_end

    if last_non_silent_sample is None:
        return audio, 0.0

    refine_start = max(0, last_non_silent_sample - frame_length)
    tail_slice = envelope[refine_start:last_non_silent_sample]
    non_silent_indices = np.flatnonzero(tail_slice > sample_threshold)
    if non_silent_indices.size > 0:
        last_non_silent_sample = refine_start + int(non_silent_indices[-1]) + 1

    keep_silence_samples = int(sample_rate * keep_silence_ms / 1000.0)
    last_sample = min(last_non_silent_sample + keep_silence_samples, audio.shape[0])
    trimmed_audio = audio[:last_sample]
    removed_seconds = max(0.0, (audio.shape[0] - trimmed_audio.shape[0]) / sample_rate)
    return trimmed_audio, removed_seconds


def trim_wav_folder(
    folder_path: Union[str, Path],
    output_folder: Optional[Union[str, Path]] = None,
    *,
    overwrite: bool = False,
    recursive: bool = True,
    top_db: float = 40.0,
    frame_length: int = 2048,
    hop_length: int = 512,
    keep_silence_ms: float = 0.0,
) -> List[Dict[str, object]]:
    """
    Trim trailing silence from every WAV file in a folder.

    If `output_folder` is not given and `overwrite` is False, files are written to
    a `trimmed/` folder inside `folder_path`. If `overwrite` is True, the original
    files are updated in place.
    """
    folder_path = Path(folder_path).expanduser()
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder_path}")
    if not folder_path.is_dir():
        raise NotADirectoryError(f"Expected a directory: {folder_path}")
    folder_path = folder_path.resolve()

    if output_folder is not None and overwrite:
        raise ValueError("Use either output_folder or overwrite=True, not both.")

    if output_folder is None:
        output_folder = folder_path if overwrite else folder_path / "trimmed"
    else:
        output_folder = Path(output_folder).expanduser().resolve()

    output_folder.mkdir(parents=True, exist_ok=True)
    output_folder_resolved = output_folder.resolve()

    search_iter = folder_path.rglob("*") if recursive else folder_path.iterdir()
    candidate_files = sorted(
        path for path in search_iter if path.is_file() and path.suffix.lower() == ".wav"
    )
    wav_files = [
        wav_path
        for wav_path in candidate_files
        if output_folder == folder_path or output_folder_resolved not in wav_path.resolve().parents
    ]

    if not wav_files:
        print(f"No WAV files found in {folder_path}")
        return []

    results: List[Dict[str, object]] = []
    for wav_path in wav_files:
        relative_path = wav_path.relative_to(folder_path)
        destination_path = output_folder / relative_path
        destination_path.parent.mkdir(parents=True, exist_ok=True)

        info = sf.info(str(wav_path))
        audio, sample_rate = sf.read(str(wav_path), always_2d=info.channels > 1)
        trimmed_audio, removed_seconds = trim_trailing_silence(
            audio,
            sample_rate,
            top_db=top_db,
            frame_length=frame_length,
            hop_length=hop_length,
            keep_silence_ms=keep_silence_ms,
        )

        changed = trimmed_audio.shape[0] < audio.shape[0]
        if changed or destination_path != wav_path:
            sf.write(str(destination_path), trimmed_audio, sample_rate, subtype=info.subtype)

        result = {
            "file": str(relative_path),
            "output": str(destination_path),
            "trimmed": changed,
            "removed_seconds": round(removed_seconds, 4),
            "original_duration_seconds": round(audio.shape[0] / sample_rate, 4),
            "trimmed_duration_seconds": round(trimmed_audio.shape[0] / sample_rate, 4),
        }
        results.append(result)

        status = "trimmed" if changed else "unchanged"
        print(f"{relative_path} -> {status} ({removed_seconds:.3f}s removed)")

    print(f"Processed {len(results)} WAV file(s).")
    return results


In [ ]:
results = trim_wav_folder(
     "./trumpet1/raw",
     overwrite=True,
     top_db=35,
     keep_silence_ms=50,
 )
results[:3]


234-1.wav -> trimmed (17.451s removed)
236-2.wav -> trimmed (7.162s removed)
236-3.wav -> trimmed (6.952s removed)
237-4.wav -> trimmed (5.451s removed)
255-2.wav -> trimmed (14.702s removed)
255-4.wav -> trimmed (6.451s removed)
257-1.wav -> trimmed (4.951s removed)
257-3.wav -> trimmed (8.952s removed)
267-3.wav -> trimmed (12.451s removed)
268-1.wav -> trimmed (5.451s removed)
269-2.wav -> unchanged (0.000s removed)
279-4.wav -> trimmed (14.546s removed)
280-1.wav -> trimmed (4.951s removed)
280-3.wav -> trimmed (10.107s removed)
282-2.wav -> trimmed (8.450s removed)
299-3.wav -> trimmed (1.285s removed)
300-1.wav -> trimmed (2.952s removed)
302-2.wav -> trimmed (6.284s removed)
315-1.wav -> trimmed (9.951s removed)
316-4.wav -> trimmed (9.549s removed)
317-2.wav -> trimmed (15.284s removed)
317-3.wav -> trimmed (4.618s removed)
333-2.wav -> trimmed (11.951s removed)
334-1.wav -> trimmed (18.618s removed)
334-5.wav -> trimmed (11.285s removed)
335-3.wav -> trimmed (13.285s removed)


[{'file': '234-1.wav',
  'output': '/Users/oriolfreixa/Documents/GitHub/GenAI/rnencodec-onehot/data/trumpet1/raw/234-1.wav',
  'trimmed': True,
  'removed_seconds': 17.451,
  'original_duration_seconds': 24.0,
  'trimmed_duration_seconds': 6.549},
 {'file': '236-2.wav',
  'output': '/Users/oriolfreixa/Documents/GitHub/GenAI/rnencodec-onehot/data/trumpet1/raw/236-2.wav',
  'trimmed': True,
  'removed_seconds': 7.1618,
  'original_duration_seconds': 24.0,
  'trimmed_duration_seconds': 16.8382},
 {'file': '236-3.wav',
  'output': '/Users/oriolfreixa/Documents/GitHub/GenAI/rnencodec-onehot/data/trumpet1/raw/236-3.wav',
  'trimmed': True,
  'removed_seconds': 6.9516,
  'original_duration_seconds': 24.0,
  'trimmed_duration_seconds': 17.0484}]

In [5]:
results = trim_wav_folder(
     "./trumpet-class/raw",
     overwrite=True,
     top_db=35,
     keep_silence_ms=50,
 )
results[:3]

234-1.wav -> trimmed (17.451s removed)
236-2.wav -> trimmed (7.162s removed)
236-3.wav -> trimmed (6.952s removed)
237-4.wav -> trimmed (5.451s removed)
255-2.wav -> trimmed (14.702s removed)
255-4.wav -> trimmed (6.451s removed)
257-1.wav -> trimmed (4.951s removed)
257-3.wav -> trimmed (8.952s removed)
267-3.wav -> trimmed (12.451s removed)
268-1.wav -> trimmed (5.451s removed)
269-2.wav -> unchanged (0.000s removed)
279-4.wav -> trimmed (14.546s removed)
280-1.wav -> trimmed (4.951s removed)
280-3.wav -> trimmed (10.107s removed)
282-2.wav -> trimmed (8.450s removed)
299-3.wav -> trimmed (1.285s removed)
300-1.wav -> trimmed (2.952s removed)
302-2.wav -> trimmed (6.284s removed)
315-1.wav -> trimmed (9.951s removed)
316-4.wav -> trimmed (9.549s removed)
317-2.wav -> trimmed (15.284s removed)
317-3.wav -> trimmed (4.618s removed)
333-2.wav -> trimmed (11.951s removed)
334-1.wav -> trimmed (18.618s removed)
334-5.wav -> trimmed (11.285s removed)
335-3.wav -> trimmed (13.285s removed)


[{'file': '234-1.wav',
  'output': '/Users/oriolfreixa/Documents/GitHub/GenAI/rnencodec-onehot/data/trumpet-class/raw/234-1.wav',
  'trimmed': True,
  'removed_seconds': 17.451,
  'original_duration_seconds': 24.0,
  'trimmed_duration_seconds': 6.549},
 {'file': '236-2.wav',
  'output': '/Users/oriolfreixa/Documents/GitHub/GenAI/rnencodec-onehot/data/trumpet-class/raw/236-2.wav',
  'trimmed': True,
  'removed_seconds': 7.1618,
  'original_duration_seconds': 24.0,
  'trimmed_duration_seconds': 16.8382},
 {'file': '236-3.wav',
  'output': '/Users/oriolfreixa/Documents/GitHub/GenAI/rnencodec-onehot/data/trumpet-class/raw/236-3.wav',
  'trimmed': True,
  'removed_seconds': 6.9516,
  'original_duration_seconds': 24.0,
  'trimmed_duration_seconds': 17.0484}]